<div align="center">

<img width="80%" src="https://user-images.githubusercontent.com/73097560/115834477-dbab4500-a447-11eb-908a-139a6edaec5c.gif"/>


<img src="https://raw.githubusercontent.com/devicons/devicon/master/icons/python/python-original.svg"
     alt="Python"
     width="80"
     height="80"/>
<img src="https://raw.githubusercontent.com/devicons/devicon/master/icons/docker/docker-original.svg"
     alt="Docker"
     width="80"
     height="80"/>
<img src="https://raw.githubusercontent.com/devicons/devicon/master/icons/scikitlearn/scikitlearn-original.svg"
     alt="scikit-learn"
     width="80"
     height="80"/>

![Python](https://img.shields.io/badge/Python-3.8+-3776AB?style=flat-square&logo=python&logoColor=white)
![Docker](https://img.shields.io/badge/Docker-2496ED?style=flat-square&logo=docker&logoColor=white)
![scikit-learn](https://img.shields.io/badge/scikit--learn-F7931E?style=flat-square&logo=scikit-learn&logoColor=white)

<h1>Docker: ML para Produção — Avaliação e Seleção de Modelos</h1>

<img width="80%" src="https://user-images.githubusercontent.com/73097560/115834477-dbab4500-a447-11eb-908a-139a6edaec5c.gif"/>

</div>

### PhD. Julles Mitoura

In [ ]:
# Obs: se você não estiver utilizando o ambiente virtual do módulo, instale as dependências:
# %pip install -q -r ../requirements.txt

---

<div align="center">

## <span style="color:#1E90FF;">Contexto</span>

</div>

Este notebook compara múltiplos algoritmos de regressão para o problema de **prever a eficiência térmica do trocador de calor** em função do tempo.

O critério de seleção vai além das métricas de treino: em ML para produção, a **capacidade de extrapolação** é fundamental. Um modelo que acerta no histórico mas falha ao projetar o futuro é inútil em produção.

Os modelos avaliados são:

| Modelo | Justificativa de inclusão |
|--------|---------------------------|
| `LinearRegression` | Hipótese principal — tendência linear observada no EDA |
| `Ridge` | Variante regularizada da regressão linear |
| `PolynomialFeatures (grau 2)` | Testa se há curvatura não-linear |
| `PolynomialFeatures (grau 3)` | Testa sobreajuste em graus mais altos |
| `RandomForestRegressor` | Benchmark de modelo não-paramétrico |

A avaliação cobre: métricas de erro, curvas de predição, capacidade de extrapolação e análise de resíduos.

In [ ]:
import os
import sqlite3

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LinearRegression, Ridge
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 100

SEED = 42
DB_PATH = os.getenv('DB_PATH', '../data/heat_exchanger.db')

---

<div align="center">

## <span style="color:#1E90FF;">Preparação dos Dados</span>

</div>

Carregamos apenas as colunas necessárias para o modelo (`timestamp` e `heat_efficiency`) e criamos o `day_index` — a feature temporal usada como input.

A divisão treino/teste é **cronológica** (sem embaralhamento): as últimas amostras formam o conjunto de teste, simulando a avaliação em dados futuros — exatamente o cenário de produção.

In [ ]:
conn = sqlite3.connect(DB_PATH)
df = pd.read_sql_query(
    'SELECT timestamp, heat_efficiency FROM heat_exchanger ORDER BY timestamp',
    conn,
)
conn.close()

df['timestamp'] = pd.to_datetime(df['timestamp'])
df['day_index'] = (df['timestamp'] - df['timestamp'].min()).dt.days

X = df[['day_index']].values
y = df['heat_efficiency'].values

# Divisão cronológica: sem shuffle — os últimos 20% são o "futuro" do modelo
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, shuffle=False
)

n_train = len(X_train)
print(f'Total   : {len(X)} amostras')
print(f'Treino  : {len(X_train)} amostras ({len(X_train)/len(X):.0%})')
print(f'Teste   : {len(X_test)} amostras ({len(X_test)/len(X):.0%})')
print(f'Período treino: {df["timestamp"].iloc[0].date()} → {df["timestamp"].iloc[n_train-1].date()}')
print(f'Período teste : {df["timestamp"].iloc[n_train].date()} → {df["timestamp"].iloc[-1].date()}')

---

<div align="center">

## <span style="color:#1E90FF;">Treinamento dos Modelos</span>

</div>

Treinamos todos os modelos candidatos no conjunto de treino e coletamos as métricas de avaliação.

O **cross-validation** com 5 folds mede a estabilidade do modelo — um R² CV com desvio padrão alto indica que o modelo é sensível à composição dos dados de treino (sinal de overfitting ou instabilidade).

In [ ]:
models = {
    'LinearRegression': LinearRegression(),
    'Ridge (α=1.0)': Ridge(alpha=1.0),
    'Polinomial grau 2': Pipeline([
        ('poly', PolynomialFeatures(degree=2, include_bias=False)),
        ('reg', LinearRegression()),
    ]),
    'Polinomial grau 3': Pipeline([
        ('poly', PolynomialFeatures(degree=3, include_bias=False)),
        ('reg', LinearRegression()),
    ]),
    'RandomForest': RandomForestRegressor(n_estimators=100, random_state=SEED),
}

results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred_test = model.predict(X_test)
    cv_scores = cross_val_score(model, X_train, y_train, cv=5, scoring='r2')
    results[name] = {
        'MAE Treino': mean_absolute_error(y_train, model.predict(X_train)),
        'MAE Teste': mean_absolute_error(y_test, y_pred_test),
        'RMSE Teste': root_mean_squared_error(y_test, y_pred_test),
        'R² Teste': r2_score(y_test, y_pred_test),
        'R² CV média': cv_scores.mean(),
        'R² CV std': cv_scores.std(),
    }
    print(f'{name:<22s} → MAE_teste={results[name]["MAE Teste"]:.4f}  R²_teste={results[name]["R² Teste"]:.4f}  R²_cv={cv_scores.mean():.4f}±{cv_scores.std():.4f}')

---

<div align="center">

## <span style="color:#1E90FF;">Comparação de Métricas</span>

</div>

A tabela abaixo consolida as métricas de todos os modelos. As células em **verde** destacam os melhores valores em cada coluna.

Atenção à diferença entre **MAE Treino** e **MAE Teste** — uma diferença grande indica overfitting.

In [ ]:
df_results = pd.DataFrame(results).T.round(4)
df_results.index.name = 'Modelo'

df_results.style \
    .highlight_min(subset=['MAE Treino', 'MAE Teste', 'RMSE Teste'], color='#d4edda') \
    .highlight_max(subset=['R² Teste', 'R² CV média'], color='#d4edda') \
    .highlight_max(subset=['R² CV std'], color='#f8d7da') \
    .format('{:.4f}')

---

<div align="center">

## <span style="color:#1E90FF;">Curvas de Predição</span>

</div>

Visualizamos as curvas de predição de cada modelo sobre o dataset completo. Modelos que passam perfeitamente pelos pontos de treino mas divergem nos pontos de teste estão sobreajustados.

A linha vertical tracejada marca o **corte treino/teste**.

In [ ]:
fig, axes = plt.subplots(1, len(models), figsize=(20, 4), sharey=True)
corte_date = df['timestamp'].iloc[n_train]

for ax, (name, model) in zip(axes, models.items()):
    y_pred = model.predict(X)
    mae_test = mean_absolute_error(y_test, model.predict(X_test))
    r2_test = r2_score(y_test, model.predict(X_test))

    ax.scatter(df['timestamp'], y, s=8, alpha=0.45, color='steelblue', label='Real', zorder=2)
    ax.plot(df['timestamp'], y_pred, color='tomato', linewidth=1.8, label='Predição')
    ax.axvline(corte_date, color='gray', linestyle='--', linewidth=1, alpha=0.7)
    ax.set_title(f'{name}\nMAE={mae_test:.4f}  R²={r2_test:.4f}', fontsize=9, fontweight='bold')
    ax.set_xlabel('Data')
    ax.tick_params(axis='x', rotation=30)
    if ax == axes[0]:
        ax.set_ylabel('Eficiência (%)')
        ax.legend(fontsize=8)

plt.suptitle('Curvas de Predição — Todos os Modelos  |  linha cinza = corte treino/teste',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

---

<div align="center">

## <span style="color:#1E90FF;">Capacidade de Extrapolação</span>

</div>

Em produção, o modelo precisa **prever além do período observado** — por exemplo, estimar a eficiência daqui a 6 meses para planejar manutenção.

Este é o critério mais importante para este problema:

- `LinearRegression` e `Ridge`: extrapolam a tendência linear indefinidamente;
- Modelos polinomiais de grau alto: podem **divergir** rapidamente fora do domínio de treino;
- `RandomForest`: **não extrapola** — retorna a média dos valores observados mais próximos.

In [ ]:
last_day = int(df['day_index'].max())
future_days = np.arange(last_day + 1, last_day + 181).reshape(-1, 1)
future_dates = df['timestamp'].max() + pd.to_timedelta(np.arange(1, 181), unit='D')

colors = ['tomato', 'steelblue', 'darkorange', 'mediumorchid', 'seagreen']

fig, ax = plt.subplots(figsize=(14, 5))

ax.scatter(df['timestamp'], y, s=8, alpha=0.35, color='gray', label='Dados históricos', zorder=2)
ax.axvline(df['timestamp'].max(), color='black', linestyle='--', linewidth=1.2,
           label='Fim do histórico')

for (name, model), color in zip(models.items(), colors):
    y_future = model.predict(future_days)
    ax.plot(future_dates, y_future, color=color, linewidth=2, label=name)

ax.set_title('Extrapolação Futura — 180 dias além do histórico', fontweight='bold', fontsize=13)
ax.set_xlabel('Data')
ax.set_ylabel('Eficiência Prevista (%)')
ax.legend(fontsize=9)
ax.tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.show()

print('Observações:')
print('  RandomForest    → plateau (retorna o valor médio dos vizinhos mais próximos)')
print('  Polinomial grau 3 → pode divergir rapidamente fora do domínio de treino')
print('  LinearRegression  → mantém a tendência linear — comportamento esperado para degradação')

---

<div align="center">

## <span style="color:#1E90FF;">Análise de Resíduos — LinearRegression</span>

</div>

Para o modelo selecionado, analisamos os resíduos (diferença entre valor real e previsto). Um bom modelo de regressão deve ter resíduos:

- **Centrados em zero** — sem viés sistemático;
- **Aleatórios ao longo do tempo** — sem padrão estrutural remanescente;
- **Distribuídos simetricamente** — próximo de uma distribuição normal.

In [ ]:
lr = models['LinearRegression']
y_pred_all = lr.predict(X)
residuals = y - y_pred_all

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# --- Resíduos ao longo do tempo ---
axes[0].scatter(df['timestamp'], residuals, s=12, alpha=0.6, color='steelblue')
axes[0].axhline(0, color='tomato', linewidth=1.5, linestyle='--')
axes[0].set_title('Resíduos ao Longo do Tempo', fontweight='bold')
axes[0].set_xlabel('Data')
axes[0].set_ylabel('Resíduo (%)')
axes[0].tick_params(axis='x', rotation=20)

# --- Distribuição dos resíduos ---
axes[1].hist(residuals, bins=25, color='steelblue', edgecolor='white', alpha=0.85)
axes[1].axvline(0, color='tomato', linewidth=1.5, linestyle='--')
axes[1].set_title(f'Distribuição dos Resíduos\n(média={residuals.mean():.4f}, std={residuals.std():.4f})',
                  fontweight='bold')
axes[1].set_xlabel('Resíduo (%)')
axes[1].set_ylabel('Frequência')

# --- Previsto vs Real ---
lims = [y.min() - 0.1, y.max() + 0.1]
axes[2].scatter(y_pred_all, y, s=12, alpha=0.6, color='steelblue')
axes[2].plot(lims, lims, color='tomato', linewidth=1.5, linestyle='--', label='Ideal (y=x)')
axes[2].set_xlim(lims)
axes[2].set_ylim(lims)
axes[2].set_title('Previsto vs Real', fontweight='bold')
axes[2].set_xlabel('Valor Previsto (%)')
axes[2].set_ylabel('Valor Real (%)')
axes[2].legend()

plt.tight_layout()
plt.show()

print(f'Resíduos — LinearRegression:')
print(f'  Média   : {residuals.mean():.6f}%  (ideal: 0)')
print(f'  Std     : {residuals.std():.4f}%')
print(f'  Mín/Máx : {residuals.min():.4f}% / {residuals.max():.4f}%')

---

<div align="center">

## <span style="color:#1E90FF;">Conclusão</span>

</div>

### Modelo selecionado: `LinearRegression`

A regressão linear é a escolha correta para este problema pelos seguintes motivos:

| Critério | LinearRegression | Polinomial grau 3 | RandomForest |
|----------|:----------------:|:-----------------:|:------------:|
| Métricas de teste competitivas | ✅ | ✅ | ✅ |
| Extrapolação linear futura | ✅ | ⚠️ diverge | ❌ plateau |
| Resíduos aleatórios sem padrão | ✅ | ✅ | ✅ |
| Interpretabilidade | ✅ coef. direto | ⚠️ | ❌ |
| Estabilidade (R² CV std baixo) | ✅ | ⚠️ | ⚠️ |

### O que vai para produção

O `train.py` treina a `LinearRegression` sobre **todo o dataset** (sem separação treino/teste) para maximizar a informação usada. A avaliação com CV garante que o modelo generaliza antes de serializar o artefato em `models/model.pkl`.

> **Princípio fundamental de ML para produção**: escolha o modelo mais simples que resolve o problema. Complexidade extra sem ganho real de performance aumenta o custo de manutenção e o risco operacional.